In [0]:
# ============================================================
# GOLD LAYER - FACT_TRANSACTIONS - UPDATED FOR CI/CD
# ============================================================

import os
from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# LOAD CONFIGURATION FROM ENVIRONMENT VARIABLES
# ============================================================

# Get environment (DEV, TEST, PROD) - Default to DEV
try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from widget: {env}")
except:
    env = os.getenv('ENV', 'DEV')
    print(f"📌 Environment from env var: {env}")

# Get storage account and access key from environment (GitHub Secrets)
storage_account_name = os.getenv('STORAGE_ACCOUNT', 'capstonestorage01')
access_key = os.getenv('STORAGE_ACCESS_KEY')

# Set container names based on environment
if env == 'DEV':
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    silver_container = 'silver'
    gold_container = 'gold'

# Configure Spark with access key (from GitHub Secrets)
if access_key:
    spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        access_key
    )
    print("✅ Storage access key configured from GitHub Secrets")
else:
    print("⚠️ WARNING: No access key found! Using Azure AD authentication")

# Dynamic paths based on environment
SILVER = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"
GOLD   = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 SILVER Container: {silver_container}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 SILVER Path: {SILVER}")
print(f"📍 GOLD Path: {GOLD}")
print(f"{'='*55}\n")

# ============================================================
# YOUR EXISTING CODE (NO CHANGES NEEDED BELOW!)
# ============================================================

# ── Load ──
df_txn      = spark.read.format("delta").load(f"{SILVER}/transactions")
dim_customer = spark.read.format("delta").load(f"{GOLD}/dim_customer")
dim_account  = spark.read.format("delta").load(f"{GOLD}/dim_account")

# ── Join & Transform ──
fact_transactions = df_txn\
    .join(dim_customer.select(
            "customer_id","risk_segment","income_band","age_group","city"),
          "customer_id", "left")\
    .join(dim_account.select(
            "account_id","credit_limit","credit_tier","rate_band","interest_rate"),
          "account_id", "left")\
    .select(
        F.col("transaction_id"),
        F.col("transaction_date"),
        F.col("account_id"),
        F.col("customer_id"),
        F.col("product_type"),
        F.col("transaction_type"),
        F.col("transaction_status"),
        F.col("amount"),
        F.col("outstanding_balance"),
        F.col("exposure"),
        F.col("interest_accrued"),
        F.col("risk_segment"),
        F.col("income_band"),
        F.col("age_group"),
        F.col("city"),
        F.col("credit_limit"),
        F.col("credit_tier"),
        F.col("rate_band"),
        F.col("risk_band"),
        F.col("year"),
        F.col("month"),
        F.round(F.col("outstanding_balance") / F.col("credit_limit") * 100, 2)
         .alias("utilization_pct"),
        F.when(F.col("transaction_status") == "Success", F.col("amount"))
         .otherwise(F.lit(0.0)).alias("successful_amount"),
        F.when(F.col("transaction_type") == "Repayment", F.col("amount"))
         .otherwise(F.lit(0.0)).alias("repayment_amount"),
        F.when(F.col("transaction_type") == "Disbursement", F.col("amount"))
         .otherwise(F.lit(0.0)).alias("disbursement_amount"),
        F.when(F.col("transaction_type") == "Fee", F.col("amount"))
         .otherwise(F.lit(0.0)).alias("fee_amount"),
        F.current_timestamp().alias("gold_created_at")
    )

# ── Write ──
fact_transactions.write\
    .format("delta")\
    .mode("overwrite")\
    .partitionBy("product_type", "year")\
    .save(f"{GOLD}/fact_transactions")

print(f"\n{'='*55}")
print(f"✅ fact_transactions : {fact_transactions.count():,} rows")
print(f"📁 Written to: {GOLD}/fact_transactions")
print(f"🏭 Environment: {env}")
print(f"{'='*55}")

✅ fact_transactions : 974,303 rows
